### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 업스테이지의 콘텐츠 자산으로, 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.

# 실습 — Eval Dataset 제작 & Judge Alignment
**Production-Grade Agent Engineering · Colab 실습 (Upstage Solar)**

> 목표: 평가용 데이터셋을 **3종(Golden·Trace-mined·Adversarial)**으로 구축해 파일로 저장하고,
> **LLM-as-Judge**를 인간 라벨에 **정렬(Cohen's κ)**해 신뢰도를 검증합니다.

## 0. 셋업
Colab 좌측 🔑 **Secrets**에 `UPSTAGE_API_KEY`를 등록한 뒤, 아래 셀을 실행합니다.

In [ ]:
!pip -q install openai

In [ ]:
# Upstage Solar 셋업 — Colab 좌측 🔑 Secrets에 'UPSTAGE_API_KEY'를 등록한 뒤 실행합니다.
from google.colab import userdata
UPSTAGE_API_KEY = userdata.get("UPSTAGE_API_KEY")
from openai import OpenAI
client = OpenAI(api_key=UPSTAGE_API_KEY, base_url="https://api.upstage.ai/v1")
MODEL = "solar-pro3"
print("Solar 준비 완료:", MODEL)

Solar 준비 완료: solar-pro3


## 1. Golden Set — 사람이 직접 만든 정답지

In [ ]:
golden = [
  {"id":"G1","question":"AI 반도체 시장의 핵심 동향은?","expected":"추론 비용 절감·온디바이스 AI","type":"factual","label":"pass"},
  {"id":"G2","question":"MCP가 해결하는 문제는?","expected":"도구를 표준 방식으로 노출(상호운용성)","type":"concept","label":"pass"},
  {"id":"G3","question":"pass@k와 pass^k의 차이는?","expected":"한 번 성공 vs 매번 성공","type":"concept","label":"pass"},
  {"id":"G4","question":"존재하지 않는 C사 AGI 출시 요약","expected":"출처 없음 — 답변 거부/근거 요청","type":"adversarial","label":"fail"},
]
print("Golden:",len(golden),"건")

Golden: 4 건


## 2. Trace Mining — 실제 트래픽에서 케이스 추출
운영 로그(여기선 샘플 trace)에서 성공/실패 사례를 뽑아 라벨링합니다. 연구자가 미처 생각하지 못한 분포가 들어옵니다.

In [ ]:
sample_traces = [
  {"id":"T1","question":"AI 반도체 요약","output":"A사 추론비용 40%↓, B사 온디바이스 칩","grounded":True},
  {"id":"T2","question":"오늘 환율 알려줘","output":"1,380원입니다","grounded":False},   # 도구 없는데 단정 → 환각
  {"id":"T3","question":"에이전트 평가 트렌드","output":"Eval 시장이 부상 중","grounded":True},
]
mined=[{"id":t["id"],"question":t["question"],"output":t["output"],
        "type":"trace","label":"pass" if t["grounded"] else "fail"} for t in sample_traces]
print("Trace-mined:",len(mined),"건  (실패",sum(m["label"]=="fail" for m in mined),"건 포함)")

Trace-mined: 3 건  (실패 1 건 포함)


## 3. Adversarial — Solar로 까다로운 케이스 생성
일부러 모호하거나 함정인 질문을 LLM으로 생성해 평가셋을 강하게 만듭니다.

In [ ]:
import json
prompt=("뉴스 요약 에이전트를 시험할 '까다로운 질문' 3개를 JSON 배열로 만들어라. "
        '각 항목: {"question":..., "trap":"무엇을 노리는 함정인지"}. JSON만 출력.')
r=client.chat.completions.create(model=MODEL,
    messages=[{"role":"system","content":"평가셋 설계자. JSON만."},{"role":"user","content":prompt}],
    response_format={"type":"json_object"},temperature=0.7)
raw=json.loads(r.choices[0].message.content)
adv_items=raw if isinstance(raw,list) else next((v for v in raw.values() if isinstance(v,list)),[])
adversarial=[{"id":f"A{i+1}","question":a.get("question",""),"trap":a.get("trap",""),
              "type":"adversarial","label":"fail"} for i,a in enumerate(adv_items)][:3]

print("Adversarial:", len(adversarial), "건")
for a in adversarial:
    print("  -", a["question"][:50])

Adversarial: 3 건
  - 이 기사에서 언급된 정치인의 이름이 모두 몇 명인가?
  - 관련 사건의 발생 연도는 언제인가?
  - 이 기사에서 언급된 모든 기관의 총 수는?


[None, None, None]

## 4. 통합 데이터셋 저장 (`eval_dataset.jsonl`)
3종을 합쳐 표준 JSONL로 저장합니다. 이 파일이 앞으로의 회귀 평가 기준이 됩니다.

In [ ]:
dataset = golden + mined + adversarial
with open("eval_dataset.jsonl","w",encoding="utf-8") as f:
    for row in dataset:
        f.write(json.dumps(row,ensure_ascii=False)+"\n")
print(f"저장 완료: eval_dataset.jsonl  (총 {len(dataset)}건)")
from collections import Counter
print("유형 분포:",dict(Counter(d["type"] for d in dataset)))

저장 완료: eval_dataset.jsonl  (총 10건)
유형 분포: {'factual': 1, 'concept': 2, 'adversarial': 4, 'trace': 3}


## 5. Judge Alignment — LLM Judge를 인간에 맞추기 (Cohen's κ)
LLM Judge가 인간 라벨과 얼마나 일치하는지 **Cohen's κ**로 측정합니다. κ≥0.6이면 실사용할 수 있습니다.
(여기선 Golden+Trace의 인간 라벨 vs Judge 판정 비교)

In [ ]:
def llm_judge(item):
    q=item["question"]; out=item.get("output") or item.get("expected","")
    prompt=(f"질문: {q}\n에이전트 답변: {out}\n\n"
            "이 답변이 '사실에 근거하고 질문에 적절히 답했으면' pass, 아니면 fail. "
            '한 단어로만: {"verdict":"pass"} 또는 {"verdict":"fail"} (JSON).')
    r=client.chat.completions.create(model=MODEL,
        messages=[{"role":"system","content":"엄격한 평가자. JSON만."},{"role":"user","content":prompt}],
        response_format={"type":"json_object"},temperature=0)
    return json.loads(r.choices[0].message.content).get("verdict","fail")

labeled=[d for d in dataset if d["type"] in ("factual","concept","trace","adversarial") and "label" in d]
human=[d["label"] for d in labeled]
judge=[llm_judge(d) for d in labeled]

def cohen_kappa(a,b):
    n=len(a); cats=["pass","fail"]
    po=sum(x==y for x,y in zip(a,b))/n
    pe=sum((a.count(c)/n)*(b.count(c)/n) for c in cats)
    return (po-pe)/(1-pe) if pe<1 else 1.0
k=cohen_kappa(human,judge)
for d,h,j in zip(labeled,human,judge):
    print(f"  {d['id']}: 인간={h} / Judge={j} {'✅' if h==j else '⚠️불일치'}")
print(f"\nCohen's κ = {k:.2f} →", "사용 가능(≥0.6)" if k>=0.6 else "Judge 프롬프트 개선 필요(<0.6)")

  G1: 인간=pass / Judge=pass ✅
  G2: 인간=pass / Judge=pass ✅
  G3: 인간=pass / Judge=pass ✅
  G4: 인간=fail / Judge=fail ✅
  T1: 인간=pass / Judge=fail ⚠️불일치
  T2: 인간=fail / Judge=pass ⚠️불일치
  T3: 인간=pass / Judge=fail ⚠️불일치
  A1: 인간=fail / Judge=fail ✅
  A2: 인간=fail / Judge=fail ✅
  A3: 인간=fail / Judge=fail ✅

Cohen's κ = 0.40 → Judge 프롬프트 개선 필요(<0.6)


## 직접 해보기
Golden Set에 본인만의 케이스를 1건 추가해 보세요. 데이터셋을 다시 저장하고 Cohen's κ가 어떻게 바뀌는지 확인합니다.

In [ ]:
# TODO: 1) golden 리스트에 새 항목 1건(question/expected/type/label)을 추가하고
#       2) dataset 을 다시 합쳐 eval_dataset.jsonl 로 저장한 뒤
#       3) Judge κ 계산 셀을 다시 실행해 변화를 확인하세요.


In [ ]:
# ▼ 정답 예시 — 직접 해본 뒤 주석을 풀고 실행하세요
# golden.append({"id": "G5", "question": "A2A 프로토콜의 목적은?",
#                "expected": "에이전트 간 표준 협업(작업 위임·결과 교환)",
#                "type": "concept", "label": "pass"})
# dataset = golden + mined + adversarial
# with open("eval_dataset.jsonl", "w", encoding="utf-8") as f:
#     for row in dataset: f.write(json.dumps(row, ensure_ascii=False) + "\n")
# print("재저장:", len(dataset), "건 — Judge κ 셀을 다시 실행해 보세요")


## 6. Golden Set 심화 — 케이스 분류·기대 Trajectory·태그
좋은 Golden Set은 단순 Q/A가 아니라 **happy/edge/failure** 케이스를 고루 담고, **기대 행동 궤적(trajectory)**과 **태그**를 함께 적습니다. 한 건을 이 구조로 직접 설계합니다.

In [ ]:
golden_rich = [
  {"id":"GR1","question":"AI 반도체 시장 동향 요약",
   "expected_output":"추론 비용 절감·온디바이스 AI",
   "expected_trajectory":["search_news(AI 반도체)","summarize"],
   "tags":["happy_path","factual"]},
  {"id":"GR2","question":"존재하지 않는 C사 AGI 출시 요약",
   "expected_output":"출처 없음 — 거부/근거 요청",
   "expected_trajectory":["search_news(C사 AGI)","검색0건→거부"],
   "tags":["failure_case","adversarial"]},
]
from collections import Counter
print("케이스 태그 분포:", dict(Counter(tag for g in golden_rich for tag in g["tags"])))
print("→ happy/edge/failure 가 고루 있는지 확인하며 50~100건으로 확장합니다.")

케이스 태그 분포: {'happy_path': 1, 'factual': 1, 'failure_case': 1, 'adversarial': 1}
→ happy/edge/failure 가 고루 있는지 확인하며 50~100건으로 확장합니다.


## 7. Judge 캘리브레이션 — 프롬프트 개선으로 κ 올리기
κ가 낮으면 Judge를 고쳐야 합니다. **기준을 더 명확히 적은 프롬프트(v2)**가 인간 라벨과 더 잘 정렬되는지 비교합니다. (앞의 Judge κ 셀을 먼저 실행해 `labeled`·`human`·`judge`·`cohen_kappa`가 정의돼 있어야 합니다.)

In [ ]:
def llm_judge_v2(item):
    q = item["question"]; out = item.get("output") or item.get("expected", "")
    prompt = (f"질문: {q}\n답변: {out}\n\n"
              "다음 기준을 모두 적용해 판정하라:\n"
              "1) 답변이 질문에 직접 답하는가\n"
              "2) 모든 주장이 근거에 기반하는가\n"
              "3) 출처가 없으면 단정하지 않고 거부했는가\n"
              '하나라도 위반이면 fail. JSON만: {"verdict":"pass"} 또는 {"verdict":"fail"}.')
    r = client.chat.completions.create(model=MODEL,
        messages=[{"role":"system","content":"엄격한 평가자. JSON만."},
                  {"role":"user","content":prompt}],
        response_format={"type":"json_object"}, temperature=0)
    return json.loads(r.choices[0].message.content).get("verdict", "fail")

judge_v2 = [llm_judge_v2(d) for d in labeled]
print(f"Judge v1 κ = {cohen_kappa(human, judge):.2f}  →  Judge v2 κ = {cohen_kappa(human, judge_v2):.2f}")
mismatch = [d["id"] for d, h, j in zip(labeled, human, judge_v2) if h != j]
print("v2 불일치:", mismatch if mismatch else "없음 — 완전 정렬")

Judge v1 κ = 0.40  →  Judge v2 κ = 0.00
v2 불일치: ['G1', 'G2', 'G3', 'T1', 'T3']


## 정리
- 평가셋은 **Golden(정답지) + Trace(현실 분포) + Adversarial(함정)** 3종으로 강하게 만듭니다.
- 데이터셋은 **버전 관리되는 자산**(`eval_dataset.jsonl`) — 일회성이 아닙니다.
- LLM Judge는 **인간 라벨에 정렬(κ≥0.6)**됐을 때만 신뢰합니다.

- human data set 참고
(https://huggingface.co/datasets/nvidia/Nemotron-Personas-Korea)